In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# -----------------------------
# 1. Load your dataset
# -----------------------------
df = pd.read_csv("../datasets/raw/combined.csv")

df = df.apply(pd.to_numeric, errors="ignore")

# -----------------------------
# 2. Choose your target variable
# -----------------------------
# Predicting entries (you can switch to exit_tap_count or overcrowding_index)
target = "entry_tap_count"

# -----------------------------
# 3. Select features
# -----------------------------
features = df.drop(columns=[
    "entry_tap_count",
    "exit_tap_count",
    "sunrise (iso8601)",
    "sunset (iso8601)",
    "time"
])

X = features
y = df[target]

# -----------------------------
# 4. Identify categorical columns
# -----------------------------
categorical_cols = ["day_of_week", "station"]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

# -----------------------------
# 5. Preprocessing
# -----------------------------
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

# -----------------------------
# 6. Build the Random Forest pipeline
# -----------------------------
model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("rf", RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ))
])

# -----------------------------
# 7. Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# 8. Fit the model
# -----------------------------
model.fit(X_train, y_train)

# -----------------------------
# 9. Evaluate
# -----------------------------
preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print("MAE:", mae)
print("R²:", r2)

# -----------------------------
# 10. Feature importances
# -----------------------------
rf = model.named_steps["rf"]
ohe = model.named_steps["preprocess"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(categorical_cols)
all_feature_names = list(cat_feature_names) + numeric_cols

importances = pd.Series(rf.feature_importances_, index=all_feature_names)
print(importances.sort_values(ascending=False).head(20))

ValueError: For a sparse output, all columns should be a numeric or convertible to a numeric.